<a href="https://colab.research.google.com/github/zelaneroz/sc4000_chatbot_area/blob/main/ChatBotArena_SC4000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LMSYS - Chatbot Arena Human Preference Predictions
[Kaggle Link](https://www.kaggle.com/competitions/lmsys-chatbot-arena/)

Nanyang Technological University
SC4000: Machine Learning
Course Project

**Members**:
* Albinowski, Wiktor Marcin (N2500433A)
* Chelashaw, Joshua Ruigu  (N2500531J)
* Espanto, Zelan Eroz (N2500107L)
* Skendri, Noussayba (N2500525K)
* Umeh, Sophia Chiarapunam (N2503901B)


**Challenge Description**
* Ensuring LLMS' responses resonate with users is critical for successful interaction.
* Predict which response a user will prefer in these head-to-head battles.

This challenge aligns with the concept of "reward models" or "preference models" in reinforcement learning from human feedback (RLHF). Previous research has identified limitations in directly prompting an existing LLM for preference predictions. These limitations often stem from biases such as favoring responses presented first (position bias), being overly verbose (verbosity bias), or exhibiting self-promotion (self-enhancement bias).



**Dataset Descrption**

Dataset collected from Chatbot Arena, where users chat with two anonymous LLMs and choose the answer they prefer.

**Evaluation**
Evaluated on the log loss between the predicted probabilities and the ground truth values (with 'eps=auto').


The submission file should contain a header and have the following format:
```
 id,winner_model_a,winner_model_b,winner_tie
 136060,0.33,0,33,0.33
 211333,0.33,0,33,0.33
 1233961,0.33,0,33,0.33
 etc
```


# 0. Imports

In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Optional
import ast
import hashlib
import math
import re

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
BASE_PATH = 'MyDrive/Acads/Spring26'
df = pd.read_csv(f'/content/drive/{BASE_PATH}/train.csv')
df.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [12]:
for i in range(10):
  print("Prompt: " + df["prompt"][i])
  print("Response A: " + df["response_a"][i])
  print("Response B: " + df["response_b"][i])

Prompt: ["Is it morally right to try to have a certain percentage of females on managerial positions?","OK, does pineapple belong on a pizza? Relax and give me fun answer."]
Response A: ["The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.\n\nHere are some arguments in favor of and against such policies:\n\n**Arguments in favor:**\n\n1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for a specific percentage can be seen as a corrective measure to address past and ongoing discrimination.\n\n2. **Promoting Diversity:** Diverse leadership teams can enhance decision-making and represent a broader range of perspectives. This can lead to better outcomes for organizations and society as a whole.\n\n3. **

#  I. Exploratory Data Analysis (EDA)

**Prelim Questions**
* How many data points are there? --> 57,477

**Additional questions**
* How many models are there and variations per model? 64
* How often are each model preferred?
* How many models tie? and which pairings often tie?
* Is there a way to differentiate the prompts? [morals, mathematics, logical, generate, etc] and is there a preference for a particular model for a particular task


In [7]:
# 1. Model Variations
all_models = pd.concat([df['model_a'], df['model_b']])
print(f"Total Unique Models: {all_models.nunique()}")
print(all_models.unique())

Total Unique Models: 64
['gpt-4-1106-preview' 'koala-13b' 'gpt-3.5-turbo-0613' 'llama-2-13b-chat'
 'vicuna-13b' 'mixtral-8x7b-instruct-v0.1' 'gemini-pro' 'gpt-4-0314'
 'vicuna-7b' 'chatglm3-6b' 'pplx-70b-online' 'mpt-30b-chat'
 'llama2-70b-steerlm-chat' 'claude-1' 'claude-2.1' 'chatglm-6b'
 'claude-instant-1' 'dolly-v2-12b' 'claude-2.0' 'deepseek-llm-67b-chat'
 'openchat-3.5' 'starling-lm-7b-alpha' 'gpt-4-0125-preview'
 'llama-2-7b-chat' 'gpt-4-0613' 'wizardlm-70b' 'stablelm-tuned-alpha-7b'
 'vicuna-33b' 'chatglm2-6b' 'dolphin-2.2.1-mistral-7b' 'llama-2-70b-chat'
 'llama-13b' 'palm-2' 'wizardlm-13b' 'codellama-34b-instruct'
 'gemini-pro-dev-api' 'gpt-3.5-turbo-0314' 'gpt-3.5-turbo-1106'
 'yi-34b-chat' 'oasst-pythia-12b' 'qwen-14b-chat' 'alpaca-13b'
 'qwen1.5-72b-chat' 'gpt-3.5-turbo-0125' 'pplx-7b-online'
 'qwen1.5-4b-chat' 'fastchat-t5-3b' 'solar-10.7b-instruct-v1.0'
 'mistral-medium' 'nous-hermes-2-mixtral-8x7b-dpo' 'zephyr-7b-beta'
 'openhermes-2.5-mistral-7b' 'mistral-7b-instruct' 

In [8]:
# 2. Preference Frequency
model_wins = pd.Series(dtype=int)
for idx, row in df.iterrows():
    if row['winner_model_a'] == 1:
        model_wins[row['model_a']] = model_wins.get(row['model_a'], 0) + 1
    elif row['winner_model_b'] == 1:
        model_wins[row['model_b']] = model_wins.get(row['model_b'], 0) + 1

# 3. Tie Analysis
tie_df = df[df['winner_tie'] == 1].copy()
tie_df['pair'] = tie_df.apply(lambda x: "-".join(sorted([x['model_a'], x['model_b']])), axis=1)
print(tie_df['pair'].value_counts())

# 4. Prompt Classification (Simple Keyword Approach)
def categorize(p):
    p = str(p).lower()
    if any(w in p for w in ['moral', 'ethic', 'right']): return 'Morals'
    if any(w in p for w in ['solve', 'math', 'calculate']): return 'Math'
    if any(w in p for w in ['logic', 'if', 'puzzle']): return 'Logic'
    if any(w in p for w in ['write', 'poem', 'create']): return 'Generation'
    return 'General'

df['category'] = df['prompt'].apply(categorize)

# # 5. Preference per Task
# df['winner'] = df.apply(lambda r: r['model_a'] if r['winner_model_a']==1 else (r['model_b'] if r['winner_model_b']==1 else 'Tie'), axis=1)
# category_pref = df[df['winner'] != 'Tie'].groupby(['category', 'winner']).size().unstack(fill_value=0)

# # Visualization
# category_pref.plot(kind='bar', stacked=True)
# plt.title('Model Preference by Task Category')
# plt.show()

pair
gpt-4-0613-gpt-4-1106-preview                   309
claude-1-claude-2.1                             292
claude-2.1-gpt-4-1106-preview                   250
gpt-4-0314-gpt-4-0613                           236
claude-2.1-gpt-4-0613                           189
                                               ... 
dolphin-2.2.1-mistral-7b-mistral-7b-instruct      1
chatglm2-6b-tulu-2-dpo-70b                        1
pplx-70b-online-vicuna-13b                        1
gpt-3.5-turbo-0125-llama2-70b-steerlm-chat        1
dolphin-2.2.1-mistral-7b-llama-2-7b-chat          1
Name: count, Length: 1172, dtype: int64


## My Understanding of the Problem
\(prompt,response A,response B)→P(A preferred)

**What's asked**
* preference prediction / reward modeling problem --> not just a classification problem, more like Pairwise ranking, Human preference modeling, RLHF reward modeling
* Evaluated using log-loss --> calibrated probabilities matter
* output: z = [z_A, z_B, z_tie]

**Log loss**
- Penalizes overconfidence
ex. prediction of [0.99 0.1 0] is bad. better to say [0.6 0.3 0.1]

**Biases**
* position bias (preference for response A)
* verbosity bias
* self-enhancement bias


## Dealing with Biases
**Positioning Bias**

Response A picked more often so:
* randomly swap A/B during training
* Add symmetric training: train on both (A,B) and (B,A)

**Verbosity Bias**

Longer responses are often preferred
* explore features like response length and token count
* normalize or explicitly model it

**Self-enhancement Bias**

LLMs prefer responses that sound more "AI-like" or confident

**Explore existing HuggingFace models**

# II. Data Preprocessing

In [16]:
from sklearn.model_selection import GroupKFold

In [15]:
import ast
import hashlib
import math
import re
from typing import Optional

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold


# -----------------------------
# 1) Text normalization helpers
# -----------------------------
def safe_literal_eval(x):
    """
    Parse string representations of Python lists safely.
    Returns x unchanged if parsing is not needed or fails.
    """
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if x.startswith("[") and x.endswith("]"):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return x
    return x


def flatten_prompt(prompt_value, sep=" [TURN_SEP] "):
    """
    Convert prompt column into a single string.
    Handles list-of-turns and plain strings.
    """
    parsed = safe_literal_eval(prompt_value)

    if isinstance(parsed, list):
        parts = [str(p).strip() for p in parsed if str(p).strip()]
        return sep.join(parts)

    return str(parsed).strip()


def clean_text(text: Optional[str]) -> str:
    """
    Minimal cleaning:
    - preserve case, punctuation, and wording
    - normalize whitespace
    - remove obvious null-like values
    """
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ""

    text = str(text)

    # If response was stored like '["..."]', optionally unwrap single-item lists
    parsed = safe_literal_eval(text)
    if isinstance(parsed, list):
        if len(parsed) == 1:
            text = str(parsed[0])
        else:
            text = " ".join(str(x) for x in parsed)

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text).strip()

    return text


# --------------------------------
# 2) Lightweight feature functions
# --------------------------------
def char_len(text: str) -> int:
    return len(text)


def word_count(text: str) -> int:
    if not text:
        return 0
    return len(re.findall(r"\S+", text))


def est_token_count(text: str) -> int:
    """
    Rough tokenizer-free estimate.
    For English-ish text, 1 token is often ~3.5 to 4.5 chars.
    We use 4 as a simple baseline.
    """
    if not text:
        return 0
    return max(1, math.ceil(len(text) / 4))


# ----------------------------------------
# 3) Target + prompt ID + fold assignment
# ----------------------------------------
def make_target_class(df: pd.DataFrame) -> pd.Series:
    """
    0 = A wins
    1 = B wins
    2 = tie
    """
    conds = [
        df["winner_model_a"].eq(1),
        df["winner_model_b"].eq(1),
        df["winner_tie"].eq(1),
    ]
    choices = [0, 1, 2]
    target = np.select(conds, choices, default=-1)

    if (target == -1).any():
        bad_rows = df.loc[target == -1, ["winner_model_a", "winner_model_b", "winner_tie"]]
        raise ValueError(
            "Found rows with invalid winner encoding. "
            "Each row should have exactly one of winner_model_a, winner_model_b, winner_tie = 1.\n"
            f"Sample bad rows:\n{bad_rows.head()}"
        )

    return pd.Series(target, index=df.index, name="target_class")


def make_prompt_id(prompt_text: str) -> str:
    """
    Stable hash-based prompt ID so the same prompt always maps to the same group.
    """
    return hashlib.md5(prompt_text.encode("utf-8")).hexdigest()


def assign_group_folds(df: pd.DataFrame, n_splits: int = 5, group_col: str = "prompt_id") -> pd.DataFrame:
    """
    GroupKFold ensures all rows from the same prompt stay in the same fold.
    This is crucial once you create swapped/symmetric duplicates.
    """
    df = df.copy()
    df["fold"] = -1

    gkf = GroupKFold(n_splits=n_splits)
    X_dummy = np.zeros(len(df))
    y_dummy = df["target_class"].values
    groups = df[group_col].values

    for fold, (_, val_idx) in enumerate(gkf.split(X_dummy, y_dummy, groups)):
        df.loc[df.index[val_idx], "fold"] = fold

    return df


# -------------------------------------
# 4) Core feature engineering function
# -------------------------------------
def add_text_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Response A features
    df["a_char_len"] = df["response_a_text"].map(char_len)
    df["a_word_count"] = df["response_a_text"].map(word_count)
    df["a_est_token_count"] = df["response_a_text"].map(est_token_count)

    # Response B features
    df["b_char_len"] = df["response_b_text"].map(char_len)
    df["b_word_count"] = df["response_b_text"].map(word_count)
    df["b_est_token_count"] = df["response_b_text"].map(est_token_count)

    # Deltas: signed and absolute
    df["char_len_delta"] = df["a_char_len"] - df["b_char_len"]
    df["word_count_delta"] = df["a_word_count"] - df["b_word_count"]
    df["est_token_count_delta"] = df["a_est_token_count"] - df["b_est_token_count"]

    df["abs_char_len_delta"] = df["char_len_delta"].abs()
    df["abs_word_count_delta"] = df["word_count_delta"].abs()
    df["abs_est_token_count_delta"] = df["est_token_count_delta"].abs()

    return df


# ------------------------------------------
# 5) Symmetric augmentation + random swapping
# ------------------------------------------
def make_symmetric_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """
    For every row, generate:
    - original orientation: (A, B)
    - swapped orientation:  (B, A)

    Labels are swapped accordingly:
    - A win <-> B win
    - tie stays tie
    """
    base = df.copy()
    base["pair_orientation"] = "original"

    swapped = df.copy()
    swapped["pair_orientation"] = "swapped"

    # Swap models
    swapped["model_a"], swapped["model_b"] = df["model_b"], df["model_a"]

    # Swap response text
    swapped["response_a_text"], swapped["response_b_text"] = df["response_b_text"], df["response_a_text"]

    # Swap one-hot labels
    swapped["winner_model_a"], swapped["winner_model_b"] = df["winner_model_b"], df["winner_model_a"]
    swapped["winner_tie"] = df["winner_tie"]

    # Swap target class: 0 <-> 1, 2 stays 2
    swapped["target_class"] = swapped["target_class"].map({0: 1, 1: 0, 2: 2})

    out = pd.concat([base, swapped], ignore_index=True)

    # Row ID for each training instance
    out["example_id"] = (
        out["id"].astype(str)
        + "_"
        + out["pair_orientation"].astype(str)
    )

    return out


def randomly_swap_pairs(
    df: pd.DataFrame,
    swap_prob: float = 0.5,
    random_state: int = 42
) -> pd.DataFrame:
    """
    Randomly swap A and B at row level and update labels/features.
    Use this when you want a stochastic training table rather than deterministic doubling.

    Recommended usage:
    - Either use symmetric pairs
    - Or use random swapping each epoch / preprocessing run
    - Or keep both options configurable
    """
    out = df.copy()
    rng = np.random.default_rng(random_state)
    swap_mask = rng.random(len(out)) < swap_prob

    # Temporary copies
    a_model = out.loc[swap_mask, "model_a"].copy()
    a_resp = out.loc[swap_mask, "response_a_text"].copy()
    a_win = out.loc[swap_mask, "winner_model_a"].copy()
    a_target = out.loc[swap_mask, "target_class"].copy()

    # Swap model columns
    out.loc[swap_mask, "model_a"] = out.loc[swap_mask, "model_b"].values
    out.loc[swap_mask, "model_b"] = a_model.values

    # Swap response text columns
    out.loc[swap_mask, "response_a_text"] = out.loc[swap_mask, "response_b_text"].values
    out.loc[swap_mask, "response_b_text"] = a_resp.values

    # Swap one-hot labels
    out.loc[swap_mask, "winner_model_a"] = out.loc[swap_mask, "winner_model_b"].values
    out.loc[swap_mask, "winner_model_b"] = a_win.values

    # Swap target class
    out.loc[swap_mask, "target_class"] = a_target.map({0: 1, 1: 0, 2: 2}).values

    out["random_swap_applied"] = swap_mask.astype(int)

    return out

In [17]:
# -----------------------------------
# 6) End-to-end preprocessing builder
# -----------------------------------
def build_preference_dataframe(
    df: pd.DataFrame,
    n_splits: int = 5,
    use_symmetric_training: bool = True,
    use_random_swap: bool = False,
    swap_prob: float = 0.5,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Build a future-proof preference modeling DataFrame.

    Output includes:
    - prompt_text
    - response_a_text / response_b_text
    - target_class
    - stable prompt_id
    - fold
    - baseline length features
    - example_id
    - orientation flags
    """

    required_cols = [
        "id", "model_a", "model_b", "prompt", "response_a", "response_b",
        "winner_model_a", "winner_model_b", "winner_tie"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    out = df.copy()

    # Normalize prompt and responses
    out["prompt_text"] = out["prompt"].map(flatten_prompt)
    out["response_a_text"] = out["response_a"].map(clean_text)
    out["response_b_text"] = out["response_b"].map(clean_text)

    # Stable single-label target
    out["target_class"] = make_target_class(out)

    # Stable prompt grouping key
    out["prompt_id"] = out["prompt_text"].map(make_prompt_id)

    # Keep traceability to original sample
    out["source_id"] = out["id"].astype(str)

    # Deterministic symmetric training
    if use_symmetric_training:
        out = make_symmetric_pairs(out)
    else:
        out["pair_orientation"] = "original"
        out["example_id"] = out["id"].astype(str) + "_original"

    # Optional stochastic swapping after that
    # In practice, many teams choose ONE of the two.
    # This is left configurable because you asked for both mechanisms.
    if use_random_swap:
        out = randomly_swap_pairs(
            out,
            swap_prob=swap_prob,
            random_state=random_state
        )
    else:
        out["random_swap_applied"] = 0

    # Recompute features after any swapping
    out = add_text_features(out)

    # Group-aware CV folds using prompt_id
    out = assign_group_folds(out, n_splits=n_splits, group_col="prompt_id")

    # Column order
    preferred_order = [
        "example_id", "source_id", "id", "prompt_id", "fold",
        "pair_orientation", "random_swap_applied",
        "model_a", "model_b",
        "prompt_text", "response_a_text", "response_b_text",
        "winner_model_a", "winner_model_b", "winner_tie", "target_class",
        "a_char_len", "a_word_count", "a_est_ntoken_count",
        "b_char_len", "b_word_count", "b_est_token_count",
        "char_len_delta", "word_count_delta", "est_token_count_delta",
        "abs_char_len_delta", "abs_word_count_delta", "abs_est_token_count_delta",
    ]

    existing_cols = [c for c in preferred_order if c in out.columns]
    remaining_cols = [c for c in out.columns if c not in existing_cols]
    out = out[existing_cols + remaining_cols]

    return out

# III. Model Training - Experimental Matrix

# IV. Model Comparison

In [ ]:
import os
import numpy as np
import pandas as pd


def run_single_setup_across_folds(processed_df, cfg, train_reward_model_fn, folds=None):
    """
    Runs one setup across all requested folds.
    Returns:
        fold_results_df: one row per fold
        summary_row: aggregated metrics for comparison table
    """
    if folds is None:
        folds = sorted(processed_df["fold"].unique().tolist())

    fold_rows = []

    for fold in folds:
        train_df = processed_df[processed_df["fold"] != fold].copy()
        val_df = processed_df[processed_df["fold"] == fold].copy()

        result = train_reward_model_fn(cfg, train_df, val_df)

        row = {
            "setup_name": cfg.name,
            "model_name": cfg.model_name,
            "objective_type": cfg.objective_type,
            "loss_type": cfg.loss_type,
            "fold": fold,
            "val_accuracy": result.get("val_accuracy", np.nan),
            "val_log_loss": result.get("val_log_loss", np.nan),
            "elo_rating": result.get("elo_rating", np.nan),
            "checkpoint_path": result.get("checkpoint_path", None),
        }
        fold_rows.append(row)

    fold_results_df = pd.DataFrame(fold_rows)

    summary_row = {
        "setup_name": cfg.name,
        "model_name": cfg.model_name,
        "objective_type": cfg.objective_type,
        "loss_type": cfg.loss_type,
        "num_folds": len(fold_results_df),
        "mean_val_accuracy": fold_results_df["val_accuracy"].mean(),
        "std_val_accuracy": fold_results_df["val_accuracy"].std(ddof=0),
        "mean_val_log_loss": fold_results_df["val_log_loss"].mean(),
        "std_val_log_loss": fold_results_df["val_log_loss"].std(ddof=0),
        "mean_elo_rating": fold_results_df["elo_rating"].mean(skipna=True),
        "std_elo_rating": fold_results_df["elo_rating"].std(ddof=0, skipna=True),
    }

    return fold_results_df, summary_row


def build_model_comparison_table(
    processed_df,
    setups,
    train_reward_model_fn,
    folds=None,
    output_dir="./comparison_outputs"
):
    """
    Executes the full experimental matrix:
    - loops over setups
    - loops over folds
    - collects per-fold metrics
    - builds aggregated model comparison table
    """
    os.makedirs(output_dir, exist_ok=True)

    all_fold_results = []
    summary_rows = []

    for setup_key, cfg in setups.items():
        print(f"\nRunning {setup_key}: {cfg.name}")

        fold_df, summary_row = run_single_setup_across_folds(
            processed_df=processed_df,
            cfg=cfg,
            train_reward_model_fn=train_reward_model_fn,
            folds=folds,
        )

        all_fold_results.append(fold_df)
        summary_rows.append(summary_row)

    all_fold_results_df = pd.concat(all_fold_results, ignore_index=True)
    comparison_df = pd.DataFrame(summary_rows)

    # sort: lower log loss is better, higher accuracy is better
    comparison_df = comparison_df.sort_values(
        by=["mean_val_log_loss", "mean_val_accuracy"],
        ascending=[True, False]
    ).reset_index(drop=True)

    # add rank columns
    comparison_df["rank_by_log_loss"] = comparison_df["mean_val_log_loss"].rank(method="min", ascending=True)
    comparison_df["rank_by_accuracy"] = comparison_df["mean_val_accuracy"].rank(method="min", ascending=False)
    comparison_df["overall_rank"] = comparison_df[["rank_by_log_loss", "rank_by_accuracy"]].mean(axis=1)
    comparison_df = comparison_df.sort_values(
        by=["overall_rank", "mean_val_log_loss"],
        ascending=[True, True]
    ).reset_index(drop=True)

    # save outputs
    fold_path = os.path.join(output_dir, "all_fold_results.csv")
    comparison_path = os.path.join(output_dir, "model_comparison_table.csv")

    all_fold_results_df.to_csv(fold_path, index=False)
    comparison_df.to_csv(comparison_path, index=False)

    return all_fold_results_df, comparison_df


def print_leaderboard(comparison_df, top_k=None):
    display_cols = [
        "setup_name",
        "model_name",
        "objective_type",
        "loss_type",
        "mean_val_log_loss",
        "std_val_log_loss",
        "mean_val_accuracy",
        "std_val_accuracy",
        "mean_elo_rating",
        "overall_rank",
    ]

    view = comparison_df[display_cols].copy()
    if top_k is not None:
        view = view.head(top_k)

    print("\n=== Model Comparison Table ===")
    print(view.to_string(index=False))


# -------------------------
# Example usage
# -------------------------
# Assumes you already have:
# 1. processed_df from your preprocessing pipeline
# 2. setups from build_experiment_matrix()
# 3. train_reward_model(config, train_df, val_df)

# setups = build_experiment_matrix()
# all_fold_results_df, comparison_df = build_model_comparison_table(
#     processed_df=processed_df,
#     setups=setups,
#     train_reward_model_fn=train_reward_model,
#     folds=[0, 1, 2, 3, 4],   # or None to use all folds in processed_df
#     output_dir="./comparison_outputs",
# )
# print_leaderboard(comparison_df, top_k=10)